# Notebook 03 — Solveurs classiques vs QAOA

**Objectif :** comparer Simulated Annealing (SA), Greedy et QAOA sur le QUBO d'assignation.

## Deux instances

| Instance | d | M | Q | Variables | Mémoire statevector | Usage |
|---|---|---|---|---|---|---|
| **Simulation** | 6 | 2 | 4 | **12** | 2¹² × 16B = 64 KB | QAOA sim locale |
| **Hardware** | 12 | 3 | 4 | **36** | 2³⁶ × 16B ≈ 1 To | IBM Torino (NB05) |

> ⚠️ **Ne jamais simuler 36 qubits en statevector** — crash garanti sur tout laptop.
> QAOA simulation = instance 12 variables. Résultats SA/Greedy = instance 36 variables (rapides).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.data.loaders import load_fred_recession_synthetic
from src.preprocessing.scaler import QuantumScaler
from src.qubo.assignment_qubo import (
    compute_marginal_alignments, build_qubo_matrix,
    decode_assignment, check_constraints,
)
from src.qubo.solvers import (
    solve_simulated_annealing, solve_greedy, solve_qaoa
)

plt.rcParams.update({'font.family': 'sans-serif', 'figure.dpi': 130, 'savefig.bbox': 'tight'})

# Données FRED (offline, pas besoin d'API pour ce notebook)
X_all, y_all, feat_all = load_fred_recession_synthetic(n_samples=200, seed=42)
X_all = QuantumScaler().fit_transform(X_all)
print(f'Données : {X_all.shape}')

## Instance A — Simulation QAOA locale (12 variables)

- d=6 features, M=2 kernels, Q=4 → **12 variables QUBO**
- Statevector : 2¹² = 4096 amplitudes → **64 KB** ✓
- Brute-force : 2¹² = 4096 solutions → validation exacte possible

In [ ]:
# Instance simulation : d=6, M=2, Q=4 → 12 variables
d_sim, M_sim, Q_sim = 6, 2, 4
kc_sim = [('ZZ', 1.0), ('XZ', 0.5)]
X_sim = X_all[:, :d_sim]

print(f'Instance simulation : d={d_sim}, M={M_sim}, Q={Q_sim}')
print(f'  Variables QUBO  : {d_sim * M_sim}')
print(f'  Statevector RAM : 2^{d_sim*M_sim} × 16B = {2**(d_sim*M_sim)*16/1024:.0f} KB ✓')
print(f'  Brute-force     : 2^{d_sim*M_sim} = {2**(d_sim*M_sim):,} solutions')

a_sim = compute_marginal_alignments(X_sim, y_all[:200], M_sim, Q_sim, kc_sim, padding='top')
Q_mat_sim = build_qubo_matrix(a_sim, d_sim, M_sim, Q_sim, lambda_div=0.5, mu1=2.0)
print(f'\nMatrice QUBO : {Q_mat_sim.shape}')

In [ ]:
# Brute-force (validation exacte)
from src.qubo.solvers import solve_brute_force

print('Brute-force (référence exacte)...')
res_bf = solve_brute_force(Q_mat_sim, d_sim, M_sim, Q_sim)
asgn_bf = decode_assignment(res_bf['x'], d_sim, M_sim, Q_sim)
valid_bf, _ = check_constraints(asgn_bf, d_sim, M_sim, Q_sim)
print(f'  Énergie optimale : {res_bf["energy"]:.4f}')
print(f'  Assignation      : {asgn_bf}')
print(f'  Valide           : {valid_bf}')

In [ ]:
# SA sur instance simulation
res_sa_sim = solve_simulated_annealing(Q_mat_sim, d_sim, M_sim, Q_sim,
                                        n_iter=20_000, seed=42)
asgn_sa_sim = decode_assignment(res_sa_sim['x'], d_sim, M_sim, Q_sim)
gap_sa = abs(res_sa_sim['energy'] - res_bf['energy'])
print(f'SA     : E={res_sa_sim["energy"]:.4f}  gap={gap_sa:.4f}  assign={asgn_sa_sim}')

# Greedy sur instance simulation
res_gr_sim = solve_greedy(Q_mat_sim, d_sim, M_sim, Q_sim)
asgn_gr_sim = decode_assignment(res_gr_sim['x'], d_sim, M_sim, Q_sim)
gap_gr = abs(res_gr_sim['energy'] - res_bf['energy'])
print(f'Greedy : E={res_gr_sim["energy"]:.4f}  gap={gap_gr:.4f}  assign={asgn_gr_sim}')

In [ ]:
# QAOA p=1 (statevector — 12 qubits, ~64 KB)
print('QAOA p=1 (statevector, 12 qubits)...')

try:
    res_qaoa1 = solve_qaoa(Q_mat_sim, d_sim, M_sim, Q_sim,
                            p=1, max_iter=200, seed=42)
    asgn_q1   = decode_assignment(res_qaoa1['x'], d_sim, M_sim, Q_sim)
    valid_q1, _ = check_constraints(asgn_q1, d_sim, M_sim, Q_sim)
    gap_q1 = abs(res_qaoa1['energy'] - res_bf['energy'])
    print(f'  E={res_qaoa1["energy"]:.4f}  gap={gap_q1:.4f}  valide={valid_q1}  assign={asgn_q1}')
    QAOA_OK = True
except Exception as e:
    print(f'  QAOA non disponible : {e}')
    print('  → pip install qiskit qiskit-aer')
    QAOA_OK = False

if QAOA_OK:
    print('\nQAOA p=2 (statevector, 12 qubits)...')
    try:
        res_qaoa2 = solve_qaoa(Q_mat_sim, d_sim, M_sim, Q_sim,
                                p=2, max_iter=300, seed=42)
        asgn_q2   = decode_assignment(res_qaoa2['x'], d_sim, M_sim, Q_sim)
        gap_q2 = abs(res_qaoa2['energy'] - res_bf['energy'])
        print(f'  E={res_qaoa2["energy"]:.4f}  gap={gap_q2:.4f}  assign={asgn_q2}')
    except Exception as e:
        print(f'  Erreur : {e}')
        res_qaoa2 = None

In [ ]:
if QAOA_OK:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Convergence QAOA
    ax = axes[0]
    ax.plot(res_qaoa1['history'], label='QAOA p=1', color='#3498db', lw=2)
    if res_qaoa2:
        ax.plot(res_qaoa2['history'], label='QAOA p=2', color='#e74c3c', lw=2)
    ax.axhline(res_bf['energy'],    ls='--', color='black', lw=1.5, label='BF optimal')
    ax.axhline(res_sa_sim['energy'], ls=':',  color='gray',  lw=1.5, label='SA')
    ax.set_xlabel('Itération COBYLA')
    ax.set_ylabel('Énergie QUBO')
    ax.set_title('Convergence QAOA\n(12 variables, statevector)')
    ax.legend(fontsize=9)

    # Comparaison gap vs optimal
    ax = axes[1]
    methods = ['Greedy', 'SA', 'QAOA p=1']
    gaps    = [gap_gr, gap_sa, gap_q1]
    colors  = ['#95a5a6', '#2ecc71', '#3498db']
    if res_qaoa2:
        methods.append('QAOA p=2')
        gaps.append(gap_q2)
        colors.append('#e74c3c')
    bars = ax.bar(methods, gaps, color=colors, alpha=0.85)
    ax.set_ylabel('Gap vs optimal (brute-force)')
    ax.set_title('Qualité des solveurs\n(12 variables QUBO, d=6, M=2, Q=4)')
    for bar, g in zip(bars, gaps):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{g:.4f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig('../results/figures/03_solver_comparison_sim.png')
    plt.show()

## Instance B — Réaliste pour IBM Torino (36 variables)

- d=12, M=3, Q=4 → **36 variables QUBO**
- Brute-force : 2³⁶ = 69 milliards → impossible
- SA et Greedy : rapides (pas de mémoire quantique)
- QAOA sur cette instance → **NB05 uniquement** (IBM Torino)

In [ ]:
d_hw, M_hw, Q_hw = 12, 3, 4
kc_hw = [('Z', 0.5), ('ZZ', 1.0), ('XZ', 0.5)]
X_hw = X_all[:, :d_hw]

print(f'Instance hardware : d={d_hw}, M={M_hw}, Q={Q_hw}')
print(f'  Variables QUBO  : {d_hw * M_hw}')
print(f'  Brute-force RAM : 2^{d_hw*M_hw} = {2**(d_hw*M_hw):,} (impossible)')
print(f'  Statevector RAM : 2^{d_hw*M_hw} × 16B ≈ {2**(d_hw*M_hw)*16/1e9:.0f} Go → CRASH')
print(f'  → QAOA seulement sur IBM Torino (NB05)')

a_hw = compute_marginal_alignments(X_hw, y_all[:200], M_hw, Q_hw, kc_hw, padding='top')
Q_mat_hw = build_qubo_matrix(a_hw, d_hw, M_hw, Q_hw, lambda_div=0.5, mu1=2.0)

print('\nSA (36 variables)...')
res_sa_hw = solve_simulated_annealing(Q_mat_hw, d_hw, M_hw, Q_hw,
                                       n_iter=50_000, seed=42)
asgn_sa_hw = decode_assignment(res_sa_hw['x'], d_hw, M_hw, Q_hw)
valid_hw, _ = check_constraints(asgn_sa_hw, d_hw, M_hw, Q_hw)
print(f'  E={res_sa_hw["energy"]:.4f}  valide={valid_hw}')
print(f'  Assignation : {asgn_sa_hw}')

print('\nGreedy (36 variables)...')
res_gr_hw = solve_greedy(Q_mat_hw, d_hw, M_hw, Q_hw)
asgn_gr_hw = decode_assignment(res_gr_hw['x'], d_hw, M_hw, Q_hw)
print(f'  E={res_gr_hw["energy"]:.4f}')

## Distribution des bitstrings QAOA (instance simulation)

In [ ]:
if QAOA_OK and 'counts' in res_qaoa1:
    counts = res_qaoa1['counts']
    sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:20]
    bitstrings, freqs = zip(*sorted_counts)

    valid_flags = []
    for bs in bitstrings:
        x_bs = np.array([int(b) for b in reversed(bs)], dtype=float)
        asgn_bs = decode_assignment(x_bs, d_sim, M_sim, Q_sim, repair=False)
        v, _ = check_constraints(asgn_bs, d_sim, M_sim, Q_sim)
        valid_flags.append(v)

    fig, ax = plt.subplots(figsize=(11, 3.5))
    colors_b = ['#2ecc71' if v else '#e74c3c' for v in valid_flags]
    ax.bar(range(len(freqs)), freqs, color=colors_b, alpha=0.85)
    ax.set_xlabel('Bitstrings (classés par fréquence)')
    ax.set_ylabel('Fréquence')
    ax.set_title(f'Distribution QAOA p=1 — 12 variables\n'
                  f'Vert = valide (Q={Q_sim} features/kernel), Rouge = invalide')
    n_valid = sum(valid_flags)
    ax.text(0.98, 0.95, f'{n_valid}/{len(valid_flags)} valides dans top-20',
            transform=ax.transAxes, ha='right', va='top')
    plt.tight_layout()
    plt.savefig('../results/figures/03_qaoa_bitstrings.png')
    plt.show()

## Récapitulatif

| Solveur | Instance | Gap optimal | Usage |
|---|---|---|---|
| Brute-force | ≤ 24 vars | 0 (exact) | Validation uniquement |
| Greedy | toutes tailles | ~10-20% | Initialisation rapide |
| SA | toutes tailles | ~1-5% | Production classique |
| QAOA p=1 | ≤ ~24 vars sim / 133 qubits HW | ~5-10% | Demo quantique |
| QAOA p=2 | ≤ ~20 vars sim / 133 qubits HW | ~2-5% | Meilleure qualité |

**Pour l'instance 36 variables (d=12, M=3) :**  
→ SA en production classique, QAOA sur IBM Torino (NB05)

**Suite → Notebook 04 : pipeline complet FRED + German Credit**